# Fetch

In [ ]:
# Creates data directory if it doesn't already exists
# !mkdir -p ./data

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta

import boto3
import pandas as pd
from ata_db_models.helpers import get_conn_string
from helpers.pathlib import PATH_DATA
from sqlalchemy import create_engine

from ata_pipeline1.fetch_events import alt_fetch_events
from ata_pipeline1.site import SITES

In [ ]:
# Start and end timestamps
# start = datetime(2022, 11, 3, 0, 0, 0)
# end = datetime(2022, 12, 8, 0, 0, 0)

# start = datetime(2022, 12, 1, 0, 0, 0)
# end = datetime(2022, 12, 29, 0, 0, 0)

start = datetime(2022, 12, 22, 0, 0, 0)
end = datetime(2023, 1, 26, 0, 0, 0)

In [ ]:
# Get database credentials
PATH_SECRETS = "/prod/ata/reader/all/database-credentials"
ssm = boto3.client("ssm")
credentials = json.loads(ssm.get_parameter(Name=PATH_SECRETS)["Parameter"]["Value"])

In [ ]:
# Set database credentials
os.environ.update({k: str(v) for (k, v) in credentials.items()})

In [ ]:
# Create engine
engine = create_engine(get_conn_string())

In [ ]:
df = pd.DataFrame()
for site in SITES.values():
    print(f"Fetching site {site.name} from {start} to {end}")
    now = time.perf_counter()
    df_fetched = alt_fetch_events(site_name=site.name, start=start, end=end, engine=engine)
    then = time.perf_counter()
    print(f"{then - now:.3f}s taken to fetch {df_fetched.shape[0]:,} rows")
    print("\n")
    df = pd.concat([df, df_fetched])

In [ ]:
# Write DataFrame to file
df.to_csv(PATH_DATA / f"{start.strftime('%y%m%d')}-{(end - timedelta(hours=1)).strftime('%y%m%d')}.csv", index=False)